In [ ]:
# @title Zelle 1: Setup, GitHub-Auth, agy CLI & OKF-Skill
import os
import subprocess
import sys

# --- Laufzeit-Check: Cloud-Abhängigkeiten absichern (Colab vs. lokal) ---
try:
    from dotenv import load_dotenv
    load_dotenv()  # lokale .env laden (falls vorhanden)
except ImportError:
    pass  # python-dotenv fehlt (z.B. in Colab); Secrets kommen dann aus google.colab.userdata
IN_COLAB = "google.colab" in sys.modules
WORKDIR = os.environ.get("NEWSCHECK_DIR") or ("/content/newscheck" if IN_COLAB else os.getcwd())

# --- 1. Git & GitHub Auth ---
subprocess.run(["git", "config", "--global", "user.name", "Colab Automation Bot"], check=True)
subprocess.run(["git", "config", "--global", "user.email", "colab-automation-bot@users.noreply.github.com"], check=True)

# --- Dual-Environment Secret Loading ---
if IN_COLAB:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
else:
    GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")
    if not GITHUB_TOKEN:
        raise ValueError("CRITICAL: GITHUB_TOKEN is missing from the local .env file!")
os.environ["GITHUB_TOKEN"] = GITHUB_TOKEN

REPO = "rhigazi/newscheck"
CLONE_URL = f"https://{GITHUB_TOKEN}@github.com/{REPO}.git"

if os.path.isdir(os.path.join(WORKDIR, ".git")):
    print("Repository vorhanden -> synchronisiere mit origin/main ...")
    # Remote-URL mit aktuellem Token setzen (robust gegen abgelaufene Tokens aus alten Clones)
    subprocess.run(["git", "-C", WORKDIR, "remote", "set-url", "origin", CLONE_URL], check=False)
    # Deterministischer Sync statt 'git pull': robust gegen divergente Branches und
    # lokale Änderungen (vermeidet Merge-Konflikte / 'divergent branches'-Fehler).
    for _cmd in (["fetch", "origin", "main"], ["reset", "--hard", "origin/main"]):
        _r = subprocess.run(["git", "-C", WORKDIR] + _cmd, capture_output=True, text=True, check=False)
        if _r.stdout.strip():
            print(_r.stdout.strip())
        if _r.returncode != 0:
            raise RuntimeError("git " + " ".join(_cmd) + " fehlgeschlagen: " + _r.stderr.strip())
else:
    print("Klone Repository ...")
    _r = subprocess.run(["git", "clone", "--depth", "1", CLONE_URL, WORKDIR], capture_output=True, text=True, check=False)
    if _r.returncode != 0:
        raise RuntimeError("git clone fehlgeschlagen: " + _r.stderr.strip())

os.chdir(WORKDIR)

# --- 2. agy CLI Installation & Pfad-Erweiterung ---
print("Prüfe agy CLI Installation...")
os.environ["PATH"] += os.pathsep + os.path.expanduser("~/.local/bin")
agy_path = subprocess.run(["which", "agy"], capture_output=True, text=True, check=False).stdout.strip()

if not agy_path:
    print("Installiere agy CLI...")
    subprocess.run("curl -fsSL https://antigravity.google/cli/install.sh | bash", shell=True, check=True)

print("Hinweis: Die agy-Authentifizierung wird vom Nutzer im Anschluss manuell durchgeführt.")

# --- 3. OKF Skill non-interaktiv installieren ---
print("Installiere OKF Skill (okf-knowledge-base)...")
env = os.environ.copy()
env["CI"] = "true"  # Zwingt npx/skills in den non-interaktiven Modus

skill_cmd = [
    "npx", "--yes", "skills", "add", 
    "https://github.com/inkeep/open-knowledge-skills", 
    "--skill", "okf-knowledge-base", 
    "--yes"
]
skill_result = subprocess.run(skill_cmd, env=env, capture_output=True, text=True, check=False)
print(skill_result.stdout)
if skill_result.returncode != 0:
    print("Hinweis bei Skill-Installation:", skill_result.stderr)

print("Setup von Repository, agy und OKF-Skill erfolgreich abgeschlossen!")


In [ ]:
# @title Zelle 2: Marktdaten-Abruf (Alpha Vantage / DIA ETF)
import os
import sys

import pandas as pd
import requests
from IPython.display import display

# --- Laufzeit-Check: Cloud-Abhängigkeiten absichern (Colab vs. lokal) ---
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass  # python-dotenv fehlt (z.B. in Colab); Secrets kommen dann aus google.colab.userdata
IN_COLAB = "google.colab" in sys.modules

# --- Dual-Environment Secret Loading ---
if IN_COLAB:
    from google.colab import userdata
    ALPHA_VANTAGE_API_KEY = userdata.get("ALPHA_VANTAGE_API_KEY")
else:
    ALPHA_VANTAGE_API_KEY = os.getenv("ALPHA_VANTAGE_API_KEY")
    if not ALPHA_VANTAGE_API_KEY:
        raise ValueError("CRITICAL: ALPHA_VANTAGE_API_KEY is missing from the local .env file!")
os.environ["ALPHA_VANTAGE_API_KEY"] = ALPHA_VANTAGE_API_KEY

SYMBOL = "DIA"  # SPDR Dow Jones Industrial Average ETF Trust (stabiler Ersatz für DJI-Index)
params = {
    "function": "TIME_SERIES_DAILY",
    "symbol": SYMBOL,
    "outputsize": "compact",
    "apikey": ALPHA_VANTAGE_API_KEY,
}

try:
    resp = requests.get("https://www.alphavantage.co/query", params=params, timeout=30)
    resp.raise_for_status()
    data = resp.json()
except requests.RequestException as e:
    raise RuntimeError("Alpha Vantage Request fehlgeschlagen: " + str(e))

# --- Rate-Limit / API-Fehler sauber abfangen ---
if "Note" in data or "Information" in data:
    msg = data.get("Note") or data.get("Information")
    raise RuntimeError("Alpha Vantage Rate-Limit erreicht: " + str(msg))
if "Error Message" in data:
    raise RuntimeError("Alpha Vantage Fehler: " + str(data["Error Message"]))
if "Time Series (Daily)" not in data:
    raise RuntimeError("Unerwartete Antwort von Alpha Vantage: " + str(list(data.keys())))

ts = data["Time Series (Daily)"]
df = pd.DataFrame.from_dict(ts, orient="index")
df.index = pd.to_datetime(df.index)
df = df.sort_index()
df.columns = ["open", "high", "low", "close", "volume"]
df = df.astype({"open": float, "high": float, "low": float, "close": float, "volume": int})
df.index.name = "date"

print("DIA-ETF-Tagesdaten erfolgreich geladen:", len(df), "Zeilen")
display(df.tail(10))


In [ ]:
# @title Zelle 3: Dynamische Tages-News-Ermittlung
import datetime
import os
import sys

IN_COLAB = "google.colab" in sys.modules
WORKDIR = os.environ.get("NEWSCHECK_DIR") or (
    "/content/newscheck" if IN_COLAB else os.getcwd()
)

WOCHENTAGE_MAP = {
    0: "montag.txt",
    1: "dienstag.txt",
    2: "mittwoch.txt",
    3: "donnerstag.txt",
    4: "freitag.txt",
    5: "samstag.txt",
}
# Wochentag ermitteln (0=Montag, ..., 5=Samstag, 6=Sonntag -> Fallback auf samstag.txt)

heute_idx = datetime.datetime.now().weekday()
ziel_datei = WOCHENTAGE_MAP.get(heute_idx, "samstag.txt")
datei_pfad = os.path.join(WORKDIR, ziel_datei)

news_archive = {}
if os.path.exists(datei_pfad):
    with open(datei_pfad, "r", encoding="utf-8", errors="replace") as f:
        news_archive[ziel_datei] = f.read()
    print(
        f"\u2705 Tagesaktuelle Datei automatisch erkannt und geladen: {ziel_datei}"
        f" ({len(news_archive[ziel_datei])} Zeichen)"
    )
else:
    raise FileNotFoundError(
        f"Die f\u00fcr heute erwartete Datei '{ziel_datei}' wurde im Pfad {WORKDIR}"
        " nicht gefunden."
    )


In [ ]:
# @title Zelle 3.5: agy-Auth-Check
import os
import subprocess

os.environ["PATH"] += os.pathsep + os.path.expanduser("~/.local/bin")

print("Pr\u00fcfe agy-Authentifizierung mit: agy -p 'Test'")
result = subprocess.run(
    ["agy", "-p", "Test"],
    capture_output=True,
    text=True,
    check=False,
    stdin=subprocess.DEVNULL,
)
if result.returncode != 0:
    print("\u26a0\ufe0f  agy ist NICHT authentifiziert. Bitte einmalig einloggen (agy starten / Browser-Login)")
    print("   und diese Zelle erneut ausf\u00fchren. Fehler:", result.stderr.strip())
else:
    print("\u2705 agy authentifiziert und bereit f\u00fcr die Analyse.")


In [ ]:
# @title Zelle 4: Prognose-Generierung & OKF-Regelsynthese via agy (OKF-Skill)
import os
import subprocess
import sys
import textwrap

# --- Laufzeit-Check: Colab-Pfad vs. lokaler Pfad ---
IN_COLAB = "google.colab" in sys.modules
WORKDIR = os.environ.get("NEWSCHECK_DIR") or ("/content/newscheck" if IN_COLAB else os.getcwd())

# --- Token-Optimierung: News-Texte vor der LLM-Übergabe begrenzen ---
MAX_CHARS_PER_FILE = 4000
MAX_TOTAL_CHARS = 20000

def _truncate(text, limit):
    text = text.strip()
    return text[:limit] + ("\n...[gekürzt]" if len(text) > limit else "")

_snippets = []
_total = 0
for _name, _text in news_archive.items():
    if _total >= MAX_TOTAL_CHARS:
        break
    _snippet = _truncate(_text, MAX_CHARS_PER_FILE)
    _snippets.append("### " + _name + "\n" + _snippet)
    _total += len(_snippet)

news_block = "\n\n".join(_snippets)
market_block = df.tail(10).to_string()

# --- Termine (Wochenvorschau) einlesen ---
_termine_path = os.path.join(WORKDIR, "termine.md")
if os.path.exists(_termine_path):
    with open(_termine_path, encoding="utf-8", errors="replace") as _f:
        termine_block = _f.read()[:4000]
else:
    termine_block = "(keine termine.md vorhanden)"

# --- Vorige Erwartung für die Soll-Ist-Validierung (Zelle 4.5) sichern ---
_erw_path = os.path.join(WORKDIR, "erwartung.md")
ERWARTUNG_VORIGE_TEXT = None
if os.path.exists(_erw_path):
    with open(_erw_path, encoding="utf-8", errors="replace") as _f:
        ERWARTUNG_VORIGE_TEXT = _f.read()

prompt = textwrap.dedent(f"""
    Nutze den installierten OKF-Skill (okf-knowledge-base) gemäß Open Knowledge Format (OKF) v0.2.

    Aufgabe 1 - OKF-Regelsynthese: Analysiere die menschlichen Verhaltensnarrative
    (z. B. Panik, Zinsangst, Erwartungshaltungen) in den Nachrichten und gleiche sie
    mit den DJIA-Kursdaten (DIA-ETF als Proxy) ab. Synthetisiere daraus OKF-Regeln und
    lege/aktualisiere die OKF-Dateien im Verzeichnis ./okf-regeln/ gemäß OKF v0.2.

    Aufgabe 2 - Prognose (erwartung.md): Erzeuge ODER aktualisiere zwingend die Datei
    "erwartung.md" im Stammverzeichnis mit einer strukturierten, tagesaktuellen
    Soll-Prognose für die anstehenden Termine. Diese Datei MUSS mit folgendem
    YAML-Frontmatter beginnen (exakte Schlüssel, maschinenlesbar):

    ---
    okf_version: "0.2"
    typ: erwartung
    datum: <YYYY-MM-DD heute>
    bezugsregel: <Dateiname der tragenden OKF-Regel ohne .md, z.B. regel-pre-event-paralyse>
    richtung: <bullisch | baerisch | neutral>
    spanne_untere: <erwarteter Unterkurs als Zahl, z.B. 508.00>
    spanne_obere: <erwarteter Oberkurs als Zahl, z.B. 516.00>
    volumen: <hoch | normal | niedrig>
    konfidenz: <Zahl 0.0-1.0>
    ---

    Danach folgt eine Prosa-Begründung inkl. der ausgewerteten Termine.

    ## Anstehende Termine (termine.md)
    {termine_block}

    ## Marktdaten (DJIA/DIA, letzte 10 Handelstage)
    {market_block}

    ## Nachrichten-Snippets (token-optimiert)
    {news_block}
""").strip()

# --- agy (Google Antigravity CLI) im Headless-/Print-Modus ausführen ---
# WICHTIG: agy ist flag-basiert (KEIN 'run'-Subcommand). --print muss das LETZTE Flag sein.
cmd = [
    "agy",
    "--dangerously-skip-permissions",
    "--add-dir", WORKDIR,
    "--print-timeout", "20m",
    "--print", prompt,
]
print("Starte agy-Prognose & OKF-Regelsynthese ...")
result = subprocess.run(
    cmd,
    capture_output=True,
    text=True,
    check=False,
    cwd=WORKDIR,
    stdin=subprocess.DEVNULL,
)
print(result.stdout)
if result.returncode != 0:
    print("agy stderr:", result.stderr)
    raise RuntimeError("agy-Analyse fehlgeschlagen")


In [ ]:
# @title Zelle 4.5: Soll-Ist-Validierung & Feedback (Robuste Regel- & Prognoseprüfung)
import datetime
import os
import sys
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

# --- Laufzeit-Check: Colab-Pfad vs. lokaler Pfad ---
IN_COLAB = "google.colab" in sys.modules
WORKDIR = os.environ.get("NEWSCHECK_DIR") or ("/content/newscheck" if IN_COLAB else os.getcwd())
OKF_DIR = os.path.join(WORKDIR, "okf-regeln")
LOG_PATH = os.path.join(OKF_DIR, "log.md")
CSV_PATH = os.path.join(OKF_DIR, "validierung_historie.csv")


def _frontmatter(text):
    """Extrahiert einfache 'schluessel: wert'-Paare aus dem YAML-Frontmatter."""
    d = {}
    lines = text.splitlines()
    if not lines or lines[0].strip() != "---":
        return d
    for line in lines[1:]:
        if line.strip() == "---":
            break
        if ":" in line:
            k, v = line.split(":", 1)
            d[k.strip().lower()] = v.strip().strip('"').strip("'")
    return d


def _adjust_konfidenz(path, delta):
    """Passt das konfidenz-Feld einer OKF-Regel an (Delta, gekappt auf 0.05..0.99)."""
    if not path or not os.path.exists(path):
        return None, None
    with open(path, encoding="utf-8") as _f:
        lines = _f.read().splitlines(keepends=True)
    idx = None
    for i, line in enumerate(lines):
        if line.strip().lower().startswith("konfidenz:"):
            idx = i
            break
    if idx is not None:
        try:
            old = float(lines[idx].split(":", 1)[1].strip())
        except ValueError:
            old = 0.50
        new = max(0.05, min(0.99, round(old + delta, 2)))
        lines[idx] = "konfidenz: " + format(new, ".2f") + "\n"
    else:
        old = 0.50
        insert_at = 1 if lines and lines[0].strip() == "---" else 0
        new = max(0.05, min(0.99, round(0.50 + delta, 2)))
        lines.insert(insert_at, "konfidenz: " + format(new, ".2f") + "\n")
    with open(path, "w", encoding="utf-8") as _f:
        _f.write("".join(lines))
    return old, new


def _annotate_rule(path, note):
    if not path or not os.path.exists(path):
        return
    with open(path, "a", encoding="utf-8") as f:
        f.write("\n> " + note + "\n")


def _append_log(entry):
    os.makedirs(OKF_DIR, exist_ok=True)
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(entry)


def _append_history_csv(row):
    os.makedirs(OKF_DIR, exist_ok=True)
    header = not os.path.exists(CSV_PATH)
    df_row = pd.DataFrame([row])
    df_row.to_csv(CSV_PATH, mode="a", header=header, index=False, encoding="utf-8")


# --- Gestern-Erwartung laden (von Zelle 4 gesichert, sonst von Disk) ---
prev_text = globals().get("ERWARTUNG_VORIGE_TEXT")
if prev_text is None:
    _p = os.path.join(WORKDIR, "erwartung.md")
    if os.path.exists(_p):
        with open(_p, encoding="utf-8", errors="replace") as _f:
            prev_text = _f.read()

if "df" not in globals():
    print("❌ Marktdaten (df) fehlen – bitte zuerst Zelle 2 ausführen. Validierung übersprungen.")
elif not prev_text:
    print("ℹ️ Keine vorige erwartung.md gefunden – erster Lauf, Validierung übersprungen.")
else:
    exp = _frontmatter(prev_text)
    bezug = exp.get("bezugsregel", "")
    rule_path = os.path.join(OKF_DIR, bezug + ".md") if bezug else None

    # --- Ist-Werte aus den DIA-Marktdaten (Zelle 2) ---
    dfv = df.sort_index()
    heute = dfv.iloc[-1]
    gestern = dfv.iloc[-2]
    close_h = float(heute["close"])
    close_g = float(gestern["close"])
    high_h = float(heute["high"])
    low_h = float(heute["low"])
    open_h = float(heute["open"])
    vol_h = float(heute["volume"])
    mean_vol = float(dfv["volume"].tail(10).mean())

    ret_pct = (close_h - close_g) / close_g
    spanne_pct = (high_h - low_h) / close_h if close_h else 0.0
    body_pct = abs(close_h - open_h) / open_h if open_h else 0.0
    vol_ratio = vol_h / mean_vol if mean_vol else 1.0

    # Realistischer Richtungs-Korridor (±0.35% als Neutralbereich statt 0.05%)
    if ret_pct > 0.0035:
        ist_richtung = "bullisch"
    elif ret_pct < -0.0035:
        ist_richtung = "baerisch"
    else:
        ist_richtung = "neutral"

    ist_volumen = "hoch" if vol_ratio > 1.25 else ("niedrig" if vol_ratio < 0.75 else "normal")

    # =========================================================================
    # TEIL 1: HYPOTHESEN-VALIDIERUNG DER BEZUGSREGEL (Verhaltensmuster)
    # =========================================================================
    rule_checks = []

    if bezug == "regel-pre-event-paralyse":
        # Erwartung: Orderbuch-Rückzug, Volumen-Dürre (< 75% Ø), Spannen-Kompression (<= 1.0%)
        c_vol = (vol_ratio < 0.75)
        c_spanne = (spanne_pct <= 0.010)
        c_body = (body_pct <= 0.006)
        rule_checks = [
            ("Volumen-Austrocknung (< 0.75x)", vol_ratio, c_vol),
            ("Spannen-Kompression (<= 1.0%)", spanne_pct * 100, c_spanne),
            ("Richtungs-Indifferenz (|Body| <= 0.6%)", body_pct * 100, c_body),
        ]
    elif bezug == "regel-relief-rally-gridlock":
        c_trend = (ret_pct >= 0.008)
        c_vol = (vol_ratio >= 1.20)
        rule_checks = [
            ("Rallye-Dynamik (>= +0.8%)", ret_pct * 100, c_trend),
            ("Volumen-Expansion (>= 1.20x)", vol_ratio, c_vol),
        ]
    elif bezug == "regel-bull-trap-realitaetsschock":
        c_reversal = ((high_h - close_h) / high_h >= 0.012) or (close_h < open_h)
        c_vol = (vol_ratio >= 1.25)
        rule_checks = [
            ("Abverkauf vom Hoch (Reversal)", (high_h - close_h) / high_h * 100, c_reversal),
            ("Panik-Volumen (>= 1.25x)", vol_ratio, c_vol),
        ]
    elif bezug == "regel-idiosynkratischer-schock-rotation":
        c_stable = (abs(ret_pct) <= 0.008)
        c_vol = (vol_ratio <= 0.85)
        rule_checks = [
            ("Index-Resilienz (|Delta| <= 0.8%)", ret_pct * 100, c_stable),
            ("Institutionelle Zurückhaltung (<= 0.85x)", vol_ratio, c_vol),
        ]
    elif bezug == "regel-zinsangst-good-news-bad-news":
        c_cap = (close_h <= high_h * 0.996)
        c_druck = (ret_pct <= 0.002)
        rule_checks = [
            ("Deckelung unter Hoch", (high_h - close_h) / high_h * 100, c_cap),
            ("Zinslast dämpft Dynamik (<= +0.2%)", ret_pct * 100, c_druck),
        ]
    elif bezug == "regel-sektordivergenz-bodenbildung":
        c_rebound = ((close_h - low_h) / low_h >= 0.004)
        rule_checks = [
            ("Bodenbildung / Intraday-Rebound (>= 0.4%)", (close_h - low_h) / low_h * 100, c_rebound),
        ]
    else:
        exp_vol = exp.get("volumen", "").lower()
        c_vol = (exp_vol == ist_volumen) if exp_vol else True
        rule_checks = [("Volumenprofil passend", vol_ratio, c_vol)]

    rule_hits = sum(1 for _, _, ok in rule_checks if ok)
    rule_total = len(rule_checks)
    # Regel gilt als bestätigt, wenn die Mehrheit der Verhaltensmerkmale eintritt
    regel_bestaetigt = (rule_hits >= max(1, (rule_total + 1) // 2))

    # =========================================================================
    # TEIL 2: PUNKTSCHÄTZUNGS-VALIDIERUNG (Taktische Tages-Prognose)
    # =========================================================================
    exp_richtung = exp.get("richtung", "").lower()
    exp_volumen = exp.get("volumen", "").lower()
    try:
        spanne_u = float(exp.get("spanne_untere", "nan"))
        spanne_o = float(exp.get("spanne_obere", "nan"))
    except ValueError:
        spanne_u = spanne_o = float("nan")

    t_richtung = (exp_richtung == ist_richtung)
    t_spanne = (spanne_u <= close_h <= spanne_o)
    t_volumen = (exp_volumen == ist_volumen)
    prog_hits = int(t_richtung) + int(t_spanne) + int(t_volumen)

    # =========================================================================
    # AUSGABE & FEEDBACK-LOOP
    # =========================================================================
    ts = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
    datum_erw = exp.get("datum", "unbekannt")

    print("=" * 72)
    print(f"1. REGEL-VALIDIERUNG (Verhaltensmuster: '{bezug}')")
    print("=" * 72)
    for name, val, ok in rule_checks:
        symbol = "✓ BESTÄTIGT" if ok else "✗ ABWEICHUNG"
        print(f"  • {name:<42} Ist={val:6.2f} -> {symbol}")
    print(f"  --> Ergebnis: {rule_hits}/{rule_total} Kriterien erfüllt.")

    if regel_bestaetigt:
        old_k, new_k = _adjust_konfidenz(rule_path, +0.05)
        print(f"  ==> REGEL BESTÄTIGT! Konfidenz erhöht: {old_k} -> {new_k} (+0.05)")
        _annotate_rule(rule_path, f"Bestätigt: {ts} Empirische Signatur ({rule_hits}/{rule_total}) am Markt verifiziert.")
        log_regel_fazit = f"Regel '{bezug}' empirisch BESTÄTIGT ({rule_hits}/{rule_total} Merkmale). Konfidenz erhöht auf {new_k}."
    else:
        old_k, new_k = _adjust_konfidenz(rule_path, -0.05)
        print(f"  ==> REGEL HINTERFRAGT. Konfidenz gesenkt: {old_k} -> {new_k} (-0.05)")
        _annotate_rule(rule_path, f"Korrektur: {ts} Signatur verfehlt ({rule_hits}/{rule_total}) – Regel hinterfragt.")
        log_regel_fazit = f"Regel '{bezug}' verfehlt ({rule_hits}/{rule_total} Merkmale). Konfidenz gesenkt auf {new_k}."

    print("\n" + "=" * 72)
    print(f"2. TAGES-PROGNOSE-VALIDIERUNG (Taktisches Kursziel vom {datum_erw})")
    print("=" * 72)
    print(f"  • Richtung : Soll={exp_richtung:<8} | Ist={ist_richtung:<8} ({ret_pct*100:+.2f}%) -> {'TREFFER' if t_richtung else 'ABWEICHUNG'}")
    print(f"  • Spanne   : Soll=[{spanne_u:.2f}, {spanne_o:.2f}] | Ist={close_h:.2f} -> {'TREFFER' if t_spanne else 'ABWEICHUNG'}")
    print(f"  • Volumen  : Soll={exp_volumen:<8} | Ist={ist_volumen:<8} (Ratio {vol_ratio:.2f}) -> {'TREFFER' if t_volumen else 'ABWEICHUNG'}")
    print(f"  --> Prognose-Score: {prog_hits}/3 Treffer.")

    # Audit in log.md schreiben
    audit_entry = (
        f"\n## Audit {ts}\n"
        f"- Bezugsregel: '{bezug}'\n"
        f"- Regel-Validierung: {'BESTÄTIGT' if regel_bestaetigt else 'ABWEICHUNG'} ({rule_hits}/{rule_total} Kriterien) -> {log_regel_fazit}\n"
        f"- Markt-Signatur: Close={close_h:.2f} ({ret_pct*100:+.2f}%), Spanne={spanne_pct*100:.2f}%, Vol-Ratio={vol_ratio:.2f} ({ist_volumen})\n"
        f"- Taktische Prognose ({datum_erw}): {prog_hits}/3 Treffer (Richtung={t_richtung}, Spanne={t_spanne}, Volumen={t_volumen})\n"
    )
    _append_log(audit_entry)

    # Strukturierte Historie in validierung_historie.csv schreiben
    hist_row = {
        "timestamp": ts,
        "erwartung_datum": datum_erw,
        "bezugsregel": bezug,
        "regel_bestaetigt": regel_bestaetigt,
        "regel_hits": f"{rule_hits}/{rule_total}",
        "prognose_score": f"{prog_hits}/3",
        "close": close_h,
        "ret_pct": round(ret_pct * 100, 2),
        "spanne_pct": round(spanne_pct * 100, 2),
        "vol_ratio": round(vol_ratio, 2),
        "richtung_ist": ist_richtung,
        "richtung_soll": exp_richtung,
        "konfidenz_neu": new_k,
    }
    _append_history_csv(hist_row)

    # Letzte 5 Historien-Einträge anzeigen
    if os.path.exists(CSV_PATH):
        print("\n" + "=" * 72)
        print("3. HISTORISCHER TRACK-RECORD (okf-regeln/validierung_historie.csv)")
        print("=" * 72)
        df_hist = pd.read_csv(CSV_PATH)
        display(df_hist.tail(5))


In [ ]:
# @title Zelle 5: Automatisierter Git Commit & Push
import datetime
import os
import subprocess
import sys

# --- Laufzeit-Check: Colab-Pfad vs. lokaler Pfad ---
IN_COLAB = "google.colab" in sys.modules
WORKDIR = os.environ.get("NEWSCHECK_DIR") or ("/content/newscheck" if IN_COLAB else os.getcwd())
os.chdir(WORKDIR)


def _git(*args):
    r = subprocess.run(["git", *args], capture_output=True, text=True, check=False)
    if r.returncode != 0:
        raise RuntimeError("git " + " ".join(args) + " fehlgeschlagen: " + (r.stderr.strip() or r.stdout.strip()))
    return r


# --- Idempotenz: nur bei echten Änderungen committen ---
# git add -A erfasst erwartung.md, okf-regeln/log.md sowie korrigierte OKF-Regeln automatisch.
status = _git("status", "--porcelain").stdout.strip()
if not status:
    print("Keine Änderungen erkannt -> nichts zu committen (idempotent).")
else:
    print("Änderungen erkannt:")
    print(status)
    _git("add", "-A")
    _stamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
    _git("commit", "-m", "chore(okf): Regelupdate & Feedback-Loop (erwartung.md/log.md) (" + _stamp + ")")

    # --- Vor dem Push mit dem neuesten origin/main synchronisieren ---
    # Robust gegen parallele Änderungen (z.B. Code-Updates auf origin): den lokalen
    # Commit auf den neuesten Stand von origin/main rebasen (Rebase-Editor deaktiviert),
    # dann fast-forward pushen. Zeigt im Fehlerfall die echte Git-Meldung an.
    _git("fetch", "origin", "main")
    _env = os.environ.copy()
    _env["GIT_EDITOR"] = "true"
    _env["GIT_SEQUENCE_EDITOR"] = "true"
    rebase = subprocess.run(["git", "rebase", "origin/main"], capture_output=True, text=True, check=False, env=_env)
    if rebase.returncode != 0:
        subprocess.run(["git", "rebase", "--abort"], capture_output=True, check=False)
        raise RuntimeError("Push nicht möglich (Rebase-Konflikt mit origin/main): " + rebase.stderr.strip())

    push = _git("push", "origin", "main")
    print("Push erfolgreich:", push.stdout or "ok")


In [ ]:
# @title Zelle 6: Telegram-Benachrichtigung (Tagesabschluss)
import datetime
import os
import sys

import requests

# --- Laufzeit-Check & Secrets (Dual-Environment wie in Zelle 1/2) ---
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass  # python-dotenv fehlt (z.B. in Colab); Secrets kommen dann aus google.colab.userdata
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import userdata
    TELEGRAM_BOT_TOKEN = userdata.get("TELEGRAM_BOT_TOKEN")
    TELEGRAM_CHAT_ID = userdata.get("TELEGRAM_CHAT_ID")
else:
    TELEGRAM_BOT_TOKEN = os.getenv("TELEGRAM_BOT_TOKEN")
    TELEGRAM_CHAT_ID = os.getenv("TELEGRAM_CHAT_ID")

if not TELEGRAM_BOT_TOKEN or not TELEGRAM_CHAT_ID:
    print("Telegram-Secrets fehlen (TELEGRAM_BOT_TOKEN / TELEGRAM_CHAT_ID) \u2013 Benachrichtigung \u00fcbersprungen.")
else:
    # Wochentag & Tagesdatei f\u00fcr die Meldung
    _tage = ["Montag", "Dienstag", "Mittwoch", "Donnerstag", "Freitag", "Samstag", "Sonntag"]
    _tag_name = _tage[datetime.datetime.now().weekday()]
    _datei = globals().get("ziel_datei", "?")
    _rows = len(df) if "df" in globals() else "?"
    _stamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")

    text = f"""\u2705 Daily Newscheck \u2013 {_tag_name} ({_datei}) erfolgreich ({_stamp})

\u2022 Marktdaten (DIA): {_rows} Zeilen
\u2022 erwartung.md: aktualisiert
\u2022 Push nach origin/main: OK"""

    # --- Telegram senden (best effort; Versandfehler sind nicht fatal) ---
    _url = f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/sendMessage"
    try:
        _resp = requests.post(_url, data={"chat_id": TELEGRAM_CHAT_ID, "text": text}, timeout=30)
        if _resp.status_code == 200 and _resp.json().get("ok"):
            print("Telegram-Nachricht gesendet \u2713")
        else:
            print("Telegram-Versand fehlgeschlagen:", _resp.status_code, _resp.text)
    except requests.RequestException as e:
        print("Telegram-Versand fehlgeschlagen (Netzwerkfehler):", e)
